In [2]:
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from torch_geometric.nn import DistMult
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
from sklearn.manifold import TSNE
from matplotlib import pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'



,id_entity_1,id_entity_2,type_entity_1,type_entity_2,predicate
638096,60877,115353,AA,SmallMolecule,interacts_with
2886279,42947,29025,AA,AA,interacts_with
2565488,39392,8380,AA,AA,interacts_with
4023465,601330,601671,SmallMolecule,SmallMolecule,has_similarity
875823,28909,37314,AA,AA,interacts_with
...,...,...,...,...,...
1075754,52139,14910,AA,AA,interacts_with
3508430,360114,360119,SmallMolecule,SmallMolecule,has_similarity
1117429,39455,61622,AA,AA,interacts_with
2807861,37606,30046,AA,AA,interacts_with


In [4]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

df = pd.read_csv('../data/edges/triples.csv')

node_keys, node_values = pd.factorize(pd.concat([df['id_entity_1'], df['id_entity_2']]))
rawid2id = {k: v.item() for v, k in zip(node_keys, node_values)}

pred_keys, pred_values = pd.factorize(df['predicate'])
pred2id = {k: v.item() for v, k in zip(pred_keys, pred_values)}

df['id_entity_1'] = df['id_entity_1'].apply(lambda x: rawid2id[x])
df['id_entity_2'] = df['id_entity_2'].apply(lambda x: rawid2id[x])
df['predicate'] = df['predicate'].apply(lambda x: pred2id[x])

hrt_arr = np.array([df['id_entity_1'].to_numpy(), df['predicate'].to_numpy(), df['id_entity_2'].to_numpy()])
hrt_tensor = torch.tensor(hrt_arr, dtype=torch.long).t()



num_triples = hrt_tensor.shape[0]

indices = torch.randperm(num_triples)
train_size = int(0.8 * num_triples)
val_size = int(0.1 * num_triples)

test_indices = indices[train_size + val_size:]
val_indices = indices[train_size : train_size + val_size]
train_indices = indices[:train_size]

train_triplets = hrt_tensor[train_indices].to(device)
val_triplets = hrt_tensor[val_indices].to(device)
test_triplets = hrt_tensor[test_indices].to(device)


filtered_dict = defaultdict(set)

# Обязательно переводим тензор на CPU и конвертируем в список Python
triplets_list = hrt_tensor.cpu().tolist()

for h, r, t in triplets_list:
    filtered_dict[(h, r)].add(t)

In [5]:
class CustomDistMult(DistMult):
    def test(
        self,
        head_index: Tensor,
        rel_type: Tensor,
        tail_index: Tensor,
        batch_size: int,
        k_list: list = [1, 5, 10, 50],
        sampling: bool = False,
        num_negs: int = 1000,
        log: bool = True,
        filtered_dict: dict = None
    ):

        arange = range(head_index.numel())
        arange = tqdm(arange) if log else arange

        mean_ranks, reciprocal_ranks, hits_at_k = [], [], {}
        for k in k_list:
            hits_at_k[k] = []

        for i in arange:
            h, r, t = head_index[i], rel_type[i], tail_index[i]
            scores = []

            if sampling:
                neg_tails = torch.randint(0, self.num_nodes, (num_negs,), device=t.device)
                tail_indices = torch.cat([t.unsqueeze(0), neg_tails])
                target_index = 0
            else:
                tail_indices = torch.arange(self.num_nodes, device=t.device)
                target_index = t

            for ts in tail_indices.split(batch_size):
                scores.append(self(h.expand_as(ts), r.expand_as(ts), ts))

            scores = torch.cat(scores)

            if filtered_dict is not None and not sampling:
                #Получаем все известные истинные хвосты для пары (h, r)
                true_tails = filtered_dict.get((int(h), int(r)), [])

                # Маскируем все известные истинные хвосты, кроме текущего целевого t
                for true_t in true_tails:
                    if true_t != int(t):
                        scores[true_t] = -float('inf') # Убираем из рейтинга


            rank = int((scores.argsort(descending=True) == target_index).nonzero().view(-1))

            mean_ranks.append(rank)
            reciprocal_ranks.append(1 / (rank + 1))
            for k in k_list:
                hits_at_k[k].append(rank < k)

        mean_rank = float(torch.tensor(mean_ranks, dtype=torch.float).mean())
        mrr = float(torch.tensor(reciprocal_ranks, dtype=torch.float).mean())
        for k in k_list:
            hits_at_k[k] = float(torch.tensor(hits_at_k[k], dtype=torch.float).mean())

        formatted_hits = {k: f"{v:.{4}f}" for k, v in hits_at_k.items()}
        return mean_rank, mrr, formatted_hits


    def loss(
        self,
        head_index: Tensor,
        rel_type: Tensor,
        tail_index: Tensor,
    ) -> Tensor:

        pos_score = self(head_index, rel_type, tail_index)
        neg_score = self(*self.sns_sample(head_index, rel_type, tail_index))

        return F.margin_ranking_loss(
            pos_score,
            neg_score,
            target=torch.ones_like(pos_score),
            margin=self.margin,
        )


    def get_sns_negatives(self, all_embs, pos_indices, n1, n2):
        """Вспомогательная функция для поиска сложных негативов"""
        # Сэмплируем N1 кандидатов для каждого триплета в батче сразу
        # shape: (num_negatives, n1)
        cand_indices = torch.randint(0, self.num_nodes, (pos_indices.size(0), n1), device=self.node_emb.weight.device)

        # Получаем эмбеддинги: позитивных сущностей и кандидатов
        pos_embs = all_embs[pos_indices].unsqueeze(1)    # (num_negatives, 1, dim)
        cand_embs = all_embs[cand_indices]                # (num_negatives, n1, dim)

        # Считаем расстояние d = ||pos - cand||
        dist = torch.norm(pos_embs - cand_embs, p=2, dim=-1) # (num_negatives, n1)

        # Считаем вероятности P = softmax(1/d)
        probs = torch.softmax(1.0 / (dist + 1e-9), dim=1)

        # Выбираем N2 лучших (самых близких) из N1
        _, top_n2_loc_idx = torch.topk(probs, k=n2, dim=1)

        # Из N2 выбираем по 1 случайному индексу для каждого примера (Exploration)
        rand_selector = torch.randint(0, n2, (pos_indices.size(0),), device=self.node_emb.weight.device)

        # Собираем финальные индексы
        final_loc_idx = top_n2_loc_idx[torch.arange(pos_indices.size(0)), rand_selector]
        return cand_indices[torch.arange(pos_indices.size(0)), final_loc_idx]



    @torch.no_grad()
    def sns_sample(
        self,
        head_index: torch.Tensor,
        rel_type: torch.Tensor,
        tail_index: torch.Tensor,
        n1: int = 50,  # Размер начального пула кандидатов
        n2: int = 5   # Размер пула "сложных" негативов
        ):
        # 1. Получаем текущие эмбеддинги всех сущностей
        # Предполагаем, что они лежат в self.node_emb.weight
        all_embs = self.node_emb.weight
        batch_size = head_index.numel()
        num_negatives = batch_size // 2

        # Клонируем индексы для модификации
        new_head = head_index.clone()
        new_tail = tail_index.clone()


        # 2. Коррептируем головы (первая половина батча)
        new_head[:num_negatives] = self.get_sns_negatives(all_embs, head_index[:num_negatives], n1, n2)

        # 3. Коррептируем хвосты (вторая половина батча)
        new_tail[num_negatives:] = self.get_sns_negatives(all_embs, tail_index[num_negatives:], n1, n2)

        return new_head, rel_type, new_tail


In [6]:
model = CustomDistMult(
    num_nodes=len(rawid2id),
    num_relations = len(pred2id),
    hidden_channels=128,
    margin=1
).to(device)

In [7]:
dataset = TensorDataset(train_triplets)
dataloader = DataLoader(dataset, batch_size=4096, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in tqdm(range(5)):
    for batch in dataloader:
        h_train = batch[0][:, 0]
        r_train = batch[0][:, 1]
        t_train = batch[0][:, 2]

        optimizer.zero_grad()
        loss = model.loss(h_train, r_train, t_train)
        loss.backward()
        optimizer.step()
    print(loss.item())


#h_test = test_triplets[:5000, 0]
#r_test = test_triplets[:5000, 1]
#t_test = test_triplets[:5000, 2]


#model.eval()
#with torch.no_grad():
#    print(model.test(h_test, r_test, t_test, 8192, filtered_dict=None, sampling=False))

 20%|██        | 1/5 [01:15<05:02, 75.75s/it]

0.02001975290477276


 40%|████      | 2/5 [02:29<03:44, 74.74s/it]

0.014130821451544762


 60%|██████    | 3/5 [03:45<02:29, 74.99s/it]

0.01677926629781723


 80%|████████  | 4/5 [04:59<01:14, 74.67s/it]

0.024215664714574814


100%|██████████| 5/5 [06:13<00:00, 74.76s/it]

0.014872502535581589


In [11]:
X_emb = model.node_emb.weight
X_emb = X_emb.detach().cpu().numpy()

In [ ]:
tsne = TSNE(n_components=2)

X_embedded = tsne.fit_transform(X_emb)